In [5]:
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

# Load variables from .env file
load_dotenv()

# Retrieve environment variables
google_api_key = os.getenv("GOOGLE_API_KEY")
langchain_api_key = os.getenv("LANGCHAIN_API_KEY")
langsmith_tracing = os.getenv("LANGSMITH_TRACING")
langsmith_endpoint = os.getenv("LANGSMITH_ENDPOINT")
langsmith_project = os.getenv("LANGSMITH_PROJECT")


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

In [6]:
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", convert_system_message_to_human=True)

In [7]:
res = model.invoke("hii").content
parser = StrOutputParser()
parser.invoke(res)

'Hi there! How can I help you today?'

In [8]:
from langchain_core.messages import HumanMessage

In [19]:
# without memory

# while True:
#     message = input("write your query:").lower()
    
#     if message == "bye":
#         print("good by have a great day!")
#         break
#     else:
#         print(parser.invoke(model.invoke([HumanMessage(content=message)])))
        
    
        

In [9]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [10]:
store = {}
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]  = InMemoryChatMessageHistory()
    return store[session_id]

In [13]:
model_with_memory = RunnableWithMessageHistory(model, get_session_history)

In [11]:
config = {"configurable" : {"session_id" : "firstchat"}}

In [14]:
model_with_memory.invoke([HumanMessage(content="hii! my name is sheryar")], config=config).content

"Hi Sheryar! It's nice to meet you. How can I help you today?"

In [15]:
# now my model can remember the chat

model_with_memory.invoke([HumanMessage(content="what is my name")], config=config).content

'Your name is Sheryar. You just told me! 😊'

In [17]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [18]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "you are a helpful assisstant. Answer all questions to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

In [19]:
chain = prompt | model

In [22]:
chain.invoke([
    HumanMessage(content="hi i am sheryar")
]).content

"Hi Sheryar! It's nice to meet you. How can I help you today?"

In [23]:
chain.invoke([
    HumanMessage(content="what is my name")
]).content

"As a large language model, I don't have access to personal information. Therefore, I don't know your name. You haven't told me!"

In [28]:
model_with_memory = RunnableWithMessageHistory(chain, get_session_history)

In [29]:
config = {"configurable" : {"session_id" : "chat3"}}

In [ ]:
response = model_with_memory.invoke(
    [HumanMessage(content="hii my name is sheryar")], 
    config=config
    
)


In [31]:
print(response)

content="Hi Sheryar, it's nice to meet you! How can I help you today?" additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []} id='run--644c343d-858c-4891-ab18-90151a8c7048-0' usage_metadata={'input_tokens': 23, 'output_tokens': 21, 'total_tokens': 44, 'input_token_details': {'cache_read': 0}}


In [25]:
chain.invoke([
    HumanMessage(content="hi i am sheryar")
]).content

"Hi Sheryar! It's nice to meet you. How can I help you today?"

In [27]:
chain.invoke([
    HumanMessage(content="what is my name")
]).content

"As a large language model, I don't have access to personal information. Therefore, I don't know your name. You haven't told me!"

In [32]:
from langchain_core.messages import SystemMessage, trim_messages


In [33]:
trimmer = trim_messages(
    max_tokens = 40,
    stretegy = "last",
    token_counter = "model",
    include_system = True,
    allow_partial = False,
    start_on = "human"
)


In [35]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

In [ ]:
chain = (
    RunnablePassthrough.assign(messages = itemgetter("messages") | trimmer)
    | prompt
    | model
    
)

response = chain.invoke(
    {
        "messages" : messages + [HumanMessage(content="whats my name")],
        "language" : "English"
    }
)

response.content